In [0]:
%sql

CREATE OR REPLACE TABLE retail_products_sql (
    product_id INT,
    product_name STRING,
    price INT
)
USING DELTA;

In [0]:
data = [
    (4, "Monitor", 12000),
    (5, "Printer", 18000),
    (6, "Webcam", 2500)
]

df = spark.createDataFrame(
    data,
    ["product_id", "product_name", "price"]
)

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail_products_pyspark")

In [0]:
display(spark.table("retail_products_pyspark"))

In [0]:
%sql

CREATE OR REPLACE TABLE retail_products_ctas
USING DELTA
AS
SELECT *
FROM VALUES
    (7, 'Tablet', 25000),
    (8, 'Speaker', 5000),
    (9, 'Router', 3500)
AS products(product_id, product_name, price);

In [0]:
%sql

SELECT * FROM retail_products_ctas;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW product_updates AS
SELECT *
FROM VALUES
    (2, 'Wireless Mouse', 950),
    (10, 'USB Hub', 1200)
AS updates(product_id, product_name, price);

In [0]:
%sql

MERGE INTO retail_products_sql AS target
USING product_updates AS source
ON target.product_id = source.product_id

WHEN MATCHED THEN
    UPDATE SET
        target.product_name = source.product_name,
        target.price = source.price

WHEN NOT MATCHED THEN
    INSERT (product_id, product_name, price)
    VALUES (source.product_id, source.product_name, source.price);

In [0]:
%sql

SELECT *
FROM retail_products_sql
ORDER BY product_id;

In [0]:
%sql

DESCRIBE HISTORY retail_products_sql;

In [0]:
%sql

DESCRIBE DETAIL retail_products_sql;

In [0]:
%sql
select * from retail_products_sql version as of 1;

In [0]:
%sql 
OPTIMIZE retail_products_sql;

In [0]:
%sql
OPTIMIZE retail_products_sql
ZORDER BY(product_id);